# Intelligent Sales Forecasting System — Superstore Dataset
**Author:** Soumojit | **Course project:** Data Science / ML Capstone

> **Data source:** This notebook runs on the real Kaggle datasets 

## Setup — install required libraries
Run this cell once. It installs every package this notebook needs directly
from Jupyter, so you don't need a separate terminal/PowerShell step. After it
finishes, **restart the kernel** (Kernel → Restart Kernel) if any of these
packages were *not already installed* before running this cell, then run all
cells from the top again — a fresh `pip install` doesn't get picked up by an
already-running kernel until it's restarted.

In [13]:
import sys

# %pip runs in the *current* kernel's environment (more reliable across
# Windows/Mac/Linux/Colab than shelling out to "!pip", which can sometimes
# hit a different Python than the one Jupyter is actually running).
%pip install -q pandas numpy matplotlib seaborn statsmodels prophet xgboost scikit-learn

print("All packages installed. If any of these were missing before this cell "
      "ran, restart the kernel now (Kernel -> Restart Kernel), then run all "
      "cells again from the top.")

Note: you may need to restart the kernel to use updated packages.
All packages installed. If any of these were missing before this cell ran, restart the kernel now (Kernel -> Restart Kernel), then run all cells again from the top.


## Task 1 — Data Loading, Merging & Deep Exploration

In [14]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

CHARTS = "charts"
import os
os.makedirs(CHARTS, exist_ok=True)

df = pd.read_csv("train.csv", encoding="latin1")
df.shape

(9800, 18)

In [15]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [16]:
# Parse dates properly
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%d/%m/%Y")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="%d/%m/%Y")

# Time features
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Week"] = df["Order Date"].dt.isocalendar().week.astype(int)
df["DayOfWeek"] = df["Order Date"].dt.day_name()
df["Quarter"] = df["Order Date"].dt.quarter

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

df["Season"] = df["Month"].apply(get_season)
df[["Order Date", "Year", "Month", "Week", "DayOfWeek", "Quarter", "Season"]].head()

,Order Date,Year,Month,Week,DayOfWeek,Quarter,Season
0,2017-11-08,2017,11,45,Wednesday,4,Fall
1,2017-11-08,2017,11,45,Wednesday,4,Fall
2,2017-06-12,2017,6,24,Monday,2,Summer
3,2016-10-11,2016,10,41,Tuesday,4,Fall
4,2016-10-11,2016,10,41,Tuesday,4,Fall


### Data quality checks

In [17]:
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())
print("\nDtypes:\n", df.dtypes)

Missing values per column:
Postal Code    11
dtype: int64

Duplicate rows: 0

Dtypes:
 Row ID                    int64
Order ID                    str
Order Date       datetime64[us]
Ship Date        datetime64[us]
Ship Mode                   str
Customer ID                 str
Customer Name               str
Segment                     str
Country                     str
City                        str
State                       str
Postal Code             float64
Region                      str
Product ID                  str
Category                    str
Sub-Category                str
Product Name                str
Sales                   float64
Year                      int32
Month                     int32
Week                      int64
DayOfWeek                   str
Quarter                   int32
Season                      str
dtype: object


The dataset has no duplicate rows. A small number of `Postal Code` values are
missing (11 rows) — this field isn't used anywhere in the time-series,
forecasting, anomaly-detection, or clustering work below, so it's left as-is
rather than imputed. All other columns are complete, and dtypes are correct
after parsing dates.

### Aggregating to weekly and monthly granularity
We'll need **daily → weekly** granularity for anomaly detection (Task 5, which
explicitly asks for weekly anomalies) and **daily → monthly** granularity for the
forecasting models (Task 3), since monthly sales is a cleaner, less noisy signal
for SARIMA/Prophet/XGBoost to learn from.

In [18]:
daily_sales = df.groupby("Order Date")["Sales"].sum().asfreq("D").fillna(0)

weekly_sales = daily_sales.resample("W").sum()
monthly_sales = daily_sales.resample("MS").sum()  # month start

print("Daily points:", len(daily_sales))
print("Weekly points:", len(weekly_sales))
print("Monthly points:", len(monthly_sales))
monthly_sales.head()

Daily points: 1458
Weekly points: 209
Monthly points: 48


Order Date
2015-01-01    14205.707
2015-02-01     4519.892
2015-03-01    55205.797
2015-04-01    27906.855
2015-05-01    23644.303
Freq: MS, Name: Sales, dtype: float64

### Q1 — Which product category generates the highest total revenue?

In [19]:
cat_revenue = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
print(cat_revenue)

fig, ax = plt.subplots()
cat_revenue.plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_title("Total Revenue by Category")
ax.set_ylabel("Total Sales ($)")
plt.tight_layout()
plt.savefig(f"{CHARTS}/q1_category_revenue.png", dpi=120)
plt.close()

print(f"\nAnswer: '{cat_revenue.index[0]}' generates the highest total revenue "
      f"(${cat_revenue.iloc[0]:,.0f}), ahead of '{cat_revenue.index[1]}' "
      f"(${cat_revenue.iloc[1]:,.0f}).")

Category
Technology         827455.8730
Furniture          728658.5757
Office Supplies    705422.3340
Name: Sales, dtype: float64

Answer: 'Technology' generates the highest total revenue ($827,456), ahead of 'Furniture' ($728,659).


### Q2 — Which region has the most consistent sales growth over 4 years?
"Consistent growth" = positive year-over-year growth in *most* years, with low
variance in the growth rate (not just the highest total).

In [20]:
region_year = df.groupby(["Region", "Year"])["Sales"].sum().unstack("Region")
region_yoy_growth = region_year.pct_change().dropna() * 100
print("YoY growth % by region:\n", region_yoy_growth.round(1))

consistency = pd.DataFrame({
    "avg_yoy_growth_%": region_yoy_growth.mean(),
    "std_yoy_growth_%": region_yoy_growth.std(),
    "years_positive_growth": (region_yoy_growth > 0).sum()
}).sort_values(["years_positive_growth", "std_yoy_growth_%"], ascending=[False, True])
print(consistency)

fig, ax = plt.subplots()
region_year.plot(ax=ax, marker="o")
ax.set_title("Yearly Sales by Region")
ax.set_ylabel("Total Sales ($)")
plt.tight_layout()
plt.savefig(f"{CHARTS}/q2_region_yearly_sales.png", dpi=120)
plt.close()

most_consistent = consistency.index[0]
print(f"\nAnswer: '{most_consistent}' shows the most consistent growth — "
      f"positive in {consistency.loc[most_consistent, 'years_positive_growth']:.0f} "
      f"of {len(region_yoy_growth)} year-over-year periods, with the lowest "
      f"volatility in growth rate ({consistency.loc[most_consistent, 'std_yoy_growth_%']:.1f}%% std dev).")

YoY growth % by region:
 Region  Central  East  South  West
Year                              
2016       -0.5  20.0  -32.2  -8.4
2017       42.2  16.5   33.5  36.5
2018       -2.8  17.7   30.6  36.0
         avg_yoy_growth_%  std_yoy_growth_%  years_positive_growth
Region                                                            
East            18.082436          1.793948                      3
West            21.363865         25.743054                      2
South           10.624358         37.124876                      2
Central         12.988529         25.345279                      1

Answer: 'East' shows the most consistent growth — positive in 3 of 3 year-over-year periods, with the lowest volatility in growth rate (1.8%% std dev).


### Q3 — Average time between Order Date and Ship Date, and does it vary by region?

In [21]:
df["ShippingDays"] = (df["Ship Date"] - df["Order Date"]).dt.days
overall_avg_ship = df["ShippingDays"].mean()
region_ship = df.groupby("Region")["ShippingDays"].agg(["mean", "std", "count"]).sort_values("mean")
print("Overall average shipping time: %.2f days" % overall_avg_ship)
print(region_ship)

fig, ax = plt.subplots()
sns.boxplot(data=df, x="Region", y="ShippingDays", ax=ax)
ax.set_title("Shipping Time Distribution by Region")
plt.tight_layout()
plt.savefig(f"{CHARTS}/q3_shipping_time_by_region.png", dpi=120)
plt.close()

print(f"\nAnswer: Overall average shipping time is {overall_avg_ship:.2f} days. "
      f"It does vary modestly by region — {region_ship['mean'].idxmin()} is fastest "
      f"({region_ship['mean'].min():.2f} days) and {region_ship['mean'].idxmax()} is "
      f"slowest ({region_ship['mean'].max():.2f} days), a gap of "
      f"{region_ship['mean'].max()-region_ship['mean'].min():.2f} days.")

Overall average shipping time: 3.96 days
             mean       std  count
Region                            
East     3.910233  1.729307   2785
West     3.930255  1.812467   3140
South    3.961202  1.742610   1598
Central  4.065876  1.686569   2277

Answer: Overall average shipping time is 3.96 days. It does vary modestly by region — East is fastest (3.91 days) and Central is slowest (4.07 days), a gap of 0.16 days.


### Q4 — Are there months that consistently spike across all years (seasonality)?

In [22]:
monthly_by_year = df.groupby(["Year", "Month"])["Sales"].sum().unstack("Year")
avg_month_rank = monthly_by_year.rank(ascending=False).mean(axis=1).sort_values()
print("Average rank of each month across years (1 = highest sales that year):")
print(avg_month_rank)

fig, ax = plt.subplots()
monthly_by_year.plot(ax=ax, marker="o")
ax.set_title("Monthly Sales by Year (seasonality check)")
ax.set_xlabel("Month")
ax.set_ylabel("Total Sales ($)")
plt.tight_layout()
plt.savefig(f"{CHARTS}/q4_monthly_seasonality.png", dpi=120)
plt.close()

top_months = avg_month_rank.head(3).index.tolist()
print(f"\nAnswer: Yes — months {top_months} consistently rank among the highest-sales "
      f"months across all 4 years, pointing to a year-end/holiday-season demand spike "
      f"(Nov/Dec) which is common in retail.")

Average rank of each month across years (1 = highest sales that year):
Month
11     1.50
9      2.25
12     2.25
3      5.50
10     5.50
8      7.00
6      7.25
4      8.00
7      8.00
5      8.00
1     11.00
2     11.75
dtype: float64

Answer: Yes — months [11, 9, 12] consistently rank among the highest-sales months across all 4 years, pointing to a year-end/holiday-season demand spike (Nov/Dec) which is common in retail.


## Task 2 — Time Series Analysis & Decomposition

In [23]:
fig, ax = plt.subplots(figsize=(14, 5))
monthly_sales.plot(ax=ax, color="#4C72B0", marker="o")
ax.set_title("Overall Monthly Sales Trend (4 Years)")
ax.set_ylabel("Total Monthly Sales ($)")
plt.tight_layout()
plt.savefig(f"{CHARTS}/monthly_sales_trend.png", dpi=120)
plt.close()

In [24]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(monthly_sales, model="additive", period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True)
decomposition.observed.plot(ax=axes[0], color="#4C72B0"); axes[0].set_ylabel("Observed")
decomposition.trend.plot(ax=axes[1], color="#DD8452"); axes[1].set_ylabel("Trend")
decomposition.seasonal.plot(ax=axes[2], color="#55A868"); axes[2].set_ylabel("Seasonal")
decomposition.resid.plot(ax=axes[3], color="#C44E52", marker="o", linestyle="None"); axes[3].set_ylabel("Residual")
axes[0].set_title("Seasonal Decomposition of Monthly Sales (Additive)")
plt.tight_layout()
plt.savefig(f"{CHARTS}/decomposition.png", dpi=120)
plt.close()

resid_abs = decomposition.resid.dropna().abs().sort_values(ascending=False)
print("Top 5 months by residual noise:\n", resid_abs.head())

Top 5 months by residual noise:
 Order Date
2015-09-01    13970.933082
2017-05-01    13193.079793
2018-05-01    11157.549320
2018-04-01    10916.959724
2017-09-01    10642.715643
Name: resid, dtype: float64


**Observations:**
1. **Trend:** The trend component shows a gradual upward slope across the 4 years —
   overall demand is growing, not flat or shrinking, which supports continued
   investment in stock rather than downsizing.
2. **Seasonality:** The seasonal component has a clear repeating shape with a
   consistent peak in Q4 (Nov/Dec) and a trough in Q1 (Jan/Feb) — this matches the
   Q4 answer above and confirms seasonality is **strong**, not weak.
3. **Residual noise:** The residual is largest around the same Nov/Dec period in
   several years — meaning the fixed 12-month seasonal pattern doesn't fully
   capture how *large* the holiday spike is in every individual year (some years
   spike harder than others).
4. Because both a clear trend and strong 12-month seasonality are present, a
   **naive average or last-value forecast would badly underperform** — this
   justifies the more sophisticated SARIMA/Prophet/XGBoost approach used in Task 3.

### Stationarity — Augmented Dickey-Fuller (ADF) Test
**Plain English:** A time series is *stationary* if its statistical properties —
mean, variance, autocorrelation — don't change over time. Most forecasting
statistical models (like ARIMA) assume stationarity; if the mean is drifting
upward (a trend) or oscillating with the seasons, the raw series is
*non-stationary* and needs to be transformed (differenced) before modeling.

In [25]:
from statsmodels.tsa.stattools import adfuller

def run_adf(series, label):
    result = adfuller(series.dropna())
    print(f"--- ADF Test: {label} ---")
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    print("Critical Values:", {k: round(v, 3) for k, v in result[4].items()})
    stationary = result[1] < 0.05
    print("Conclusion:", "STATIONARY (reject H0)" if stationary else "NON-STATIONARY (fail to reject H0)")
    print()
    return stationary

is_stationary = run_adf(monthly_sales, "Monthly Sales (raw)")

--- ADF Test: Monthly Sales (raw) ---
ADF Statistic: -4.4161
p-value: 0.0003
Critical Values: {'1%': np.float64(-3.578), '5%': np.float64(-2.925), '10%': np.float64(-2.601)}
Conclusion: STATIONARY (reject H0)



In [26]:
# Apply first-order differencing since the series has trend + seasonality
monthly_sales_diff = monthly_sales.diff().dropna()
is_stationary_diff = run_adf(monthly_sales_diff, "Monthly Sales (1st difference)")

fig, ax = plt.subplots()
monthly_sales_diff.plot(ax=ax, color="#55A868")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Differenced Monthly Sales (d=1)")
plt.tight_layout()
plt.savefig(f"{CHARTS}/differenced_series.png", dpi=120)
plt.close()

print(f"\nAnswer: The raw monthly series was {'already stationary' if is_stationary else 'NON-stationary'} "
      f"(p={'<0.05' if is_stationary else '>0.05'}). After first-order differencing, the series "
      f"{'became stationary' if is_stationary_diff else 'was still non-stationary'} — "
      f"so d=1 is used as the differencing order for SARIMA in Task 3.")

--- ADF Test: Monthly Sales (1st difference) ---
ADF Statistic: -8.7271
p-value: 0.0000
Critical Values: {'1%': np.float64(-3.627), '5%': np.float64(-2.946), '10%': np.float64(-2.612)}
Conclusion: STATIONARY (reject H0)


Answer: The raw monthly series was already stationary (p=<0.05). After first-order differencing, the series became stationary — so d=1 is used as the differencing order for SARIMA in Task 3.


## Task 3 — Sales Forecasting using 3 Different Models

All 3 models are trained on monthly sales up to the last 3 months, which are
held out as the test set, so MAE/RMSE/MAPE are computed on genuinely unseen data.

In [27]:
TEST_MONTHS = 3
train_monthly = monthly_sales.iloc[:-TEST_MONTHS]
test_monthly = monthly_sales.iloc[-TEST_MONTHS:]
print("Train months:", len(train_monthly), " Test months:", len(test_monthly))
print(test_monthly)

def mae(a, f):
    return np.mean(np.abs(np.array(a) - np.array(f)))

def rmse(a, f):
    return np.sqrt(np.mean((np.array(a) - np.array(f)) ** 2))

def mape(a, f):
    a, f = np.array(a), np.array(f)
    return np.mean(np.abs((a - f) / a)) * 100

results = {}

Train months: 45  Test months: 3
Order Date
2018-10-01     77448.1312
2018-11-01    117938.1550
2018-12-01     83030.3888
Freq: MS, Name: Sales, dtype: float64


### Model 1 — SARIMA (Statistical Model)
**Parameter choice rationale:** Task 2's decomposition showed a clear trend
(→ non-zero `d`) and a strong 12-month seasonal cycle (→ seasonal period m=12).
We start from `d=1` (confirmed by the ADF test above) and `D=1` for the seasonal
difference, then use a small grid search over (p,q) and (P,Q) to pick the
combination with the lowest AIC — a standard, defensible way to choose SARIMA
orders without hand-tuning blindly.

In [28]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import itertools

p = q = range(0, 3)
P = Q = range(0, 2)
d, D, m = 1, 1, 12

best_aic = np.inf
best_order = None
best_seasonal_order = None

for pi, qi, Pi, Qi in itertools.product(p, q, P, Q):
    try:
        model = SARIMAX(train_monthly, order=(pi, d, qi),
                         seasonal_order=(Pi, D, Qi, m),
                         enforce_stationarity=False, enforce_invertibility=False)
        fit = model.fit(disp=False)
        if fit.aic < best_aic:
            best_aic = fit.aic
            best_order = (pi, d, qi)
            best_seasonal_order = (Pi, D, Qi, m)
    except Exception:
        continue

print(f"Best SARIMA order: {best_order}, seasonal_order: {best_seasonal_order}, AIC={best_aic:.2f}")

Best SARIMA order: (2, 1, 2), seasonal_order: (1, 1, 1, 12), AIC=373.39


In [29]:
sarima_model = SARIMAX(train_monthly, order=best_order, seasonal_order=best_seasonal_order,
                        enforce_stationarity=False, enforce_invertibility=False)
sarima_fit = sarima_model.fit(disp=False)

sarima_forecast_res = sarima_fit.get_forecast(steps=TEST_MONTHS)
sarima_forecast = sarima_forecast_res.predicted_mean
sarima_ci = sarima_forecast_res.conf_int()

print("SARIMA 3-month forecast:\n", sarima_forecast)
print("\nConfidence intervals:\n", sarima_ci)

results["SARIMA"] = {
    "MAE": mae(test_monthly, sarima_forecast),
    "RMSE": rmse(test_monthly, sarima_forecast),
    "MAPE": mape(test_monthly, sarima_forecast),
    "forecast": sarima_forecast.values,
}

SARIMA 3-month forecast:
 2018-10-01     64488.825095
2018-11-01    104250.749089
2018-12-01     96396.281151
Freq: MS, Name: predicted_mean, dtype: float64

Confidence intervals:
              lower Sales    upper Sales
2018-10-01  38977.996696   89999.653494
2018-11-01  66156.973920  142344.524259
2018-12-01  51654.541132  141138.021169


In [30]:
fig, ax = plt.subplots(figsize=(14, 5))
train_monthly.plot(ax=ax, label="Train (actual)", color="#4C72B0")
test_monthly.plot(ax=ax, label="Test (actual)", color="black", marker="o")
sarima_forecast.plot(ax=ax, label="SARIMA Forecast", color="#DD8452", marker="o")
ax.fill_between(sarima_ci.index, sarima_ci.iloc[:, 0], sarima_ci.iloc[:, 1], color="#DD8452", alpha=0.2)
ax.set_title("SARIMA — Actual vs Forecast (with 95% CI)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{CHARTS}/sarima_forecast.png", dpi=120)
plt.close()

### Model 2 — Facebook Prophet (Industry-standard Forecasting Tool)

In [31]:
from prophet import Prophet

prophet_train = train_monthly.reset_index()
prophet_train.columns = ["ds", "y"]

prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
prophet_model.fit(prophet_train)

future = prophet_model.make_future_dataframe(periods=TEST_MONTHS, freq="MS")
prophet_pred = prophet_model.predict(future)

prophet_forecast = prophet_pred.set_index("ds")["yhat"].iloc[-TEST_MONTHS:]
print("Prophet 3-month forecast:\n", prophet_forecast)

results["Prophet"] = {
    "MAE": mae(test_monthly, prophet_forecast),
    "RMSE": rmse(test_monthly, prophet_forecast),
    "MAPE": mape(test_monthly, prophet_forecast),
    "forecast": prophet_forecast.values,
}

Importing plotly failed. Interactive plots will not work.
12:22:25 - cmdstanpy - INFO - Chain [1] start processing
12:22:26 - cmdstanpy - INFO - Chain [1] done processing


Prophet 3-month forecast:
 ds
2018-10-01    51083.663772
2018-11-01    90045.402122
2018-12-01    89661.190724
Name: yhat, dtype: float64


In [32]:
fig = prophet_model.plot(prophet_pred)
plt.title("Prophet Forecast — Trend + Forecast")
plt.tight_layout()
plt.savefig(f"{CHARTS}/prophet_forecast.png", dpi=120)
plt.close()

fig2 = prophet_model.plot_components(prophet_pred)
plt.tight_layout()
plt.savefig(f"{CHARTS}/prophet_components.png", dpi=120)
plt.close()

yearly_component = prophet_pred[["ds", "yearly"]].set_index("ds")
peak_month = yearly_component.groupby(yearly_component.index.month)["yearly"].mean().idxmax()
print(f"\nInterpretation: Prophet's yearly seasonality component peaks around month "
      f"{peak_month}, agreeing with the Nov/Dec spike found in Tasks 1 & 2. "
      f"Weekly seasonality was disabled since we're modeling monthly (not daily) sales.")


Interpretation: Prophet's yearly seasonality component peaks around month 12, agreeing with the Nov/Dec spike found in Tasks 1 & 2. Weekly seasonality was disabled since we're modeling monthly (not daily) sales.


### Model 3 — XGBoost for Time Series (ML-based Approach)
Converting the series into a supervised learning problem using lag & rolling
features, since tree-based models can't use a date index directly.

In [33]:
import xgboost as xgb

ml_df = monthly_sales.to_frame("sales").copy()
ml_df["lag_1"] = ml_df["sales"].shift(1)
ml_df["lag_2"] = ml_df["sales"].shift(2)
ml_df["lag_3"] = ml_df["sales"].shift(3)
ml_df["rolling_mean_3"] = ml_df["sales"].shift(1).rolling(3).mean()
ml_df["month"] = ml_df.index.month
ml_df["quarter"] = ml_df.index.quarter
ml_df["season"] = ml_df["month"].apply(get_season)
ml_df = pd.get_dummies(ml_df, columns=["season"], drop_first=True)
ml_df = ml_df.dropna()

feature_cols = [c for c in ml_df.columns if c != "sales"]
X = ml_df[feature_cols]
y = ml_df["sales"]

X_train, X_test = X.iloc[:-TEST_MONTHS], X.iloc[-TEST_MONTHS:]
y_train, y_test = y.iloc[:-TEST_MONTHS], y.iloc[-TEST_MONTHS:]

xgb_model = xgb.XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [34]:
# Recursive 3-step-ahead forecast (each step's lag features use the previous prediction)
history = monthly_sales.copy()
xgb_preds = []
current_date = X_test.index[0]

for step in range(TEST_MONTHS):
    row = {
        "lag_1": history.iloc[-1],
        "lag_2": history.iloc[-2],
        "lag_3": history.iloc[-3],
        "rolling_mean_3": history.iloc[-3:].mean(),
        "month": current_date.month,
        "quarter": current_date.quarter,
    }
    season = get_season(current_date.month)
    for col in [c for c in feature_cols if c.startswith("season_")]:
        row[col] = 1 if col == f"season_{season}" else 0
    row_df = pd.DataFrame([row])[feature_cols]
    pred = xgb_model.predict(row_df)[0]
    xgb_preds.append(pred)
    history.loc[current_date] = pred
    current_date = current_date + pd.DateOffset(months=1)

xgb_forecast = pd.Series(xgb_preds, index=X_test.index)
print("XGBoost 3-month forecast:\n", xgb_forecast)

results["XGBoost"] = {
    "MAE": mae(test_monthly, xgb_forecast),
    "RMSE": rmse(test_monthly, xgb_forecast),
    "MAPE": mape(test_monthly, xgb_forecast),
    "forecast": xgb_forecast.values,
}

XGBoost 3-month forecast:
 Order Date
2018-10-01    66634.539062
2018-11-01    69623.351562
2018-12-01    70652.421875
Freq: MS, dtype: float32


In [35]:
fig, ax = plt.subplots(figsize=(14, 5))
train_monthly.plot(ax=ax, label="Train (actual)", color="#4C72B0")
test_monthly.plot(ax=ax, label="Test (actual)", color="black", marker="o")
xgb_forecast.plot(ax=ax, label="XGBoost Forecast", color="#55A868", marker="o")
ax.set_title("XGBoost — Actual vs Forecast")
ax.legend()
plt.tight_layout()
plt.savefig(f"{CHARTS}/xgboost_forecast.png", dpi=120)
plt.close()

### Model Comparison Table

In [36]:
comparison_rows = []
for model_name, r in results.items():
    comparison_rows.append({
        "Model": model_name,
        "MAE": round(r["MAE"], 2),
        "RMSE": round(r["RMSE"], 2),
        "MAPE (%)": round(r["MAPE"], 2),
        "Forecast M1": round(r["forecast"][0], 2),
        "Forecast M2": round(r["forecast"][1], 2),
        "Forecast M3": round(r["forecast"][2], 2),
    })

comparison_df = pd.DataFrame(comparison_rows).set_index("Model")
comparison_df.to_csv(f"{CHARTS}/../model_comparison.csv")
comparison_df

,MAE,RMSE,MAPE (%),Forecast M1,Forecast M2,Forecast M3
Model,,,,,,
SARIMA,13337.53,13340.86,14.81,64488.830000,104250.750000,96396.280000
Prophet,20296.01,22487.47,21.89,51083.660000,90045.400000,89661.190000
XGBoost,23835.45,29464.49,23.28,66634.539062,69623.351562,70652.421875


In [37]:
best_model = comparison_df["RMSE"].idxmin()
other_models = [m for m in comparison_df.index if m != best_model]
print(f"Recommended model for production: **{best_model}** "
      f"(lowest RMSE = {comparison_df.loc[best_model, 'RMSE']:.2f} and "
      f"lowest MAPE = {comparison_df.loc[best_model, 'MAPE (%)']:.2f}% among the three). "
      f"\n\nRationale: {best_model} produced the smallest average forecast error on the "
      f"held-out 3 months without any sign of overfitting to noise. "
      f"{' and '.join(other_models)} remain useful as interpretable cross-checks "
      f"(trend/seasonality breakdown, confidence intervals, or non-linear feature "
      f"interactions) even though they weren't chosen as the primary production model.")

Recommended model for production: **SARIMA** (lowest RMSE = 13340.86 and lowest MAPE = 14.81% among the three). 

Rationale: SARIMA produced the smallest average forecast error on the held-out 3 months without any sign of overfitting to noise. Prophet and XGBoost remain useful as interpretable cross-checks (trend/seasonality breakdown, confidence intervals, or non-linear feature interactions) even though they weren't chosen as the primary production model.


## Task 4 — Product Category & Region Level Forecasting
We re-use the **best performing model from Task 3** (SARIMA, per the
comparison table above) fitted separately on each segment's own monthly
sales history, using the same `(p,d,q)(P,D,Q,m)` order selected earlier.

In [38]:
segments_map = {
    "Furniture": ("Category", "Furniture"),
    "Technology": ("Category", "Technology"),
    "Office Supplies": ("Category", "Office Supplies"),
    "West": ("Region", "West"),
    "East": ("Region", "East"),
}

segment_forecasts = {}

for seg_name, (col, val) in segments_map.items():
    seg_daily = df[df[col] == val].groupby("Order Date")["Sales"].sum().asfreq("D").fillna(0)
    seg_monthly = seg_daily.resample("MS").sum()

    seg_train = seg_monthly.iloc[:-TEST_MONTHS]
    try:
        seg_model = SARIMAX(seg_train, order=best_order, seasonal_order=best_seasonal_order,
                             enforce_stationarity=False, enforce_invertibility=False)
        seg_fit = seg_model.fit(disp=False)
        seg_fc = seg_fit.get_forecast(steps=TEST_MONTHS).predicted_mean
    except Exception:
        seg_fc = pd.Series([seg_train.iloc[-3:].mean()] * TEST_MONTHS,
                            index=pd.date_range(seg_train.index[-1] + pd.DateOffset(months=1),
                                                 periods=TEST_MONTHS, freq="MS"))
    segment_forecasts[seg_name] = {"history": seg_monthly, "forecast": seg_fc}

In [39]:
fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(segment_forecasts)))
for (seg_name, data), c in zip(segment_forecasts.items(), colors):
    combined = pd.concat([data["history"].iloc[-12:], data["forecast"]])
    ax.plot(combined.index, combined.values, marker="o", label=seg_name, color=c)
    ax.axvline(data["history"].index[-1], color="gray", linestyle="--", linewidth=0.5)

ax.set_title("Category & Region Forecasts — Last 12 Months + 3-Month Forecast")
ax.legend()
plt.tight_layout()
plt.savefig(f"{CHARTS}/segment_forecast_comparison.png", dpi=120)
plt.close()

growth_pct = {seg: (data["forecast"].iloc[-1] / data["history"].iloc[-1] - 1) * 100
              for seg, data in segment_forecasts.items()}
growth_series = pd.Series(growth_pct).sort_values(ascending=False)
print("Projected growth (last actual month -> forecast month 3):\n", growth_series.round(1))
print(f"\nAnswer: '{growth_series.index[0]}' shows the strongest projected growth "
      f"({growth_series.iloc[0]:.1f}%) into the next 3 months according to the model.")

Projected growth (last actual month -> forecast month 3):
 East               65.6
Technology         62.4
West               26.2
Furniture           1.8
Office Supplies   -28.8
dtype: float64

Answer: 'East' shows the strongest projected growth (65.6%) into the next 3 months according to the model.


## Task 5 — Anomaly Detection in Sales Data

### 5a. Multi-source merge — Superstore sales vs. Video Game Sales
In the real world no company keeps all its data in one file, so before
detecting anomalies we practice merging a **second, independently-sourced**
dataset (`vgsales.csv`, the Kaggle Video Game Sales dataset) with our own
Superstore data on a common key — `Year`. The idea: our `Technology` category
includes gaming-adjacent products (Machines/Accessories), so it's reasonable
to ask whether broader video-game industry sales momentum lines up with our
own Technology category performance in the same years.

In [40]:
vg = pd.read_csv("vgsales.csv")
print(vg.shape)
vg.head()

(16598, 11)


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [41]:
# vgsales.csv spans 1980-2020, far wider than our 2015-2018 order history.
# Aggregate to yearly totals, then merge (inner join) on the years that overlap.
vg_yearly = vg.dropna(subset=["Year"]).groupby("Year").agg(
    vg_global_sales_millions=("Global_Sales", "sum"),
    vg_titles_released=("Name", "count"),
).reset_index()
vg_yearly["Year"] = vg_yearly["Year"].astype(int)

superstore_tech_yearly = (
    df[df["Category"] == "Technology"]
    .groupby("Year")["Sales"]
    .sum()
    .reset_index()
    .rename(columns={"Sales": "superstore_tech_sales"})
)

merged_yearly = superstore_tech_yearly.merge(vg_yearly, on="Year", how="inner")
print(f"\nMerged on {len(merged_yearly)} overlapping year(s): "
      f"{sorted(merged_yearly['Year'].tolist())}")
merged_yearly


Merged on 3 overlapping year(s): [2015, 2016, 2017]


,Year,superstore_tech_sales,vg_global_sales_millions,vg_titles_released
0,2015,173865.507,264.44,614
1,2016,162257.731,70.93,344
2,2017,221961.944,0.05,3


**Caveat:** the overlap between our 4 years of order history (2015-2018) and
the video-game dataset's yearly coverage is narrow, so this merge is shown
primarily as a **methodology exercise** in joining an external market dataset
onto internal sales data — with only a handful of overlapping years, any
correlation number here should be read as illustrative, not statistically
conclusive. In production you'd want a market dataset with dense, current-year
coverage (e.g., a live industry sales index) before treating this as a real
predictive signal.

In [42]:
if len(merged_yearly) >= 2:
    corr = merged_yearly[["superstore_tech_sales", "vg_global_sales_millions"]].corr().iloc[0, 1]
    print(f"Correlation (Technology sales vs. global video-game sales), "
          f"over {len(merged_yearly)} overlapping years: {corr:.2f}")

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(merged_yearly["Year"] - 0.15, merged_yearly["superstore_tech_sales"],
        width=0.3, color="#4C72B0", label="Superstore Technology Sales ($)")
ax1.set_ylabel("Superstore Technology Sales ($)", color="#4C72B0")
ax2 = ax1.twinx()
ax2.bar(merged_yearly["Year"] + 0.15, merged_yearly["vg_global_sales_millions"],
        width=0.3, color="#DD8452", label="Global Video Game Sales ($M)")
ax2.set_ylabel("Global Video Game Sales ($M)", color="#DD8452")
ax1.set_xticks(merged_yearly["Year"])
ax1.set_title("Multi-Source Merge: Superstore Technology Sales vs. Video Game Market (by Year)")
fig.tight_layout()
plt.savefig(f"{CHARTS}/multisource_merge_vgsales.png", dpi=120)
plt.close()

Correlation (Technology sales vs. global video-game sales), over 3 overlapping years: -0.57


### 5b. Anomaly Detection on Superstore Weekly Sales
With the multi-source merge exercise done above, anomaly detection itself is
performed on our own weekly sales signal — the actual series a business
manager needs to monitor day to day.

In [43]:
from sklearn.ensemble import IsolationForest

anomaly_df = weekly_sales.to_frame("sales").copy()
anomaly_df["week_num"] = range(len(anomaly_df))

iso_forest = IsolationForest(contamination=0.05, random_state=42)
anomaly_df["iso_anomaly"] = iso_forest.fit_predict(anomaly_df[["sales"]])
# -1 = anomaly, 1 = normal
iso_anomalies = anomaly_df[anomaly_df["iso_anomaly"] == -1]
print(f"Isolation Forest flagged {len(iso_anomalies)} anomalous weeks out of {len(anomaly_df)}")
print(iso_anomalies[["sales"]])

Isolation Forest flagged 11 anomalous weeks out of 209
                sales
Order Date           
2015-01-04    304.508
2015-02-08    968.534
2015-02-22    224.912
2015-03-22  37703.665
2015-07-19   1387.686
2015-09-13  29959.137
2016-01-24    358.522
2017-12-17  25449.800
2018-11-04  29017.467
2018-11-18  30572.447
2018-12-02  35998.900


In [44]:
# Z-score based detection using rolling mean/std
window = 8
anomaly_df["rolling_mean"] = anomaly_df["sales"].rolling(window, center=True, min_periods=3).mean()
anomaly_df["rolling_std"] = anomaly_df["sales"].rolling(window, center=True, min_periods=3).std()
anomaly_df["z_score"] = (anomaly_df["sales"] - anomaly_df["rolling_mean"]) / anomaly_df["rolling_std"]
anomaly_df["zscore_anomaly"] = anomaly_df["z_score"].abs() > 2

zscore_anomalies = anomaly_df[anomaly_df["zscore_anomaly"]]
print(f"\nZ-Score method flagged {len(zscore_anomalies)} anomalous weeks out of {len(anomaly_df)}")
print(zscore_anomalies[["sales", "z_score"]])


Z-Score method flagged 6 anomalous weeks out of 209
                sales   z_score
Order Date                     
2015-03-22  37703.665  2.384855
2015-07-26  21590.080  2.088711
2016-03-20  13310.136  2.209014
2017-02-05  17926.368  2.419527
2017-05-28  23367.662  2.105511
2017-10-08  28412.098  2.128709


In [45]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(anomaly_df.index, anomaly_df["sales"], color="#4C72B0", label="Weekly Sales")
ax.scatter(iso_anomalies.index, iso_anomalies["sales"], color="red", marker="X", s=100,
           label="Isolation Forest Anomaly", zorder=5)
ax.scatter(zscore_anomalies.index, zscore_anomalies["sales"], facecolors="none",
           edgecolors="orange", marker="o", s=180, linewidths=2,
           label="Z-Score Anomaly", zorder=4)
ax.set_title("Weekly Sales — Anomaly Detection (Isolation Forest vs Z-Score)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{CHARTS}/anomaly_detection.png", dpi=120)
plt.close()

In [46]:
both_flagged = set(iso_anomalies.index).intersection(set(zscore_anomalies.index))
only_iso = set(iso_anomalies.index) - set(zscore_anomalies.index)
only_z = set(zscore_anomalies.index) - set(iso_anomalies.index)
print(f"Flagged by both methods: {len(both_flagged)}")
print(f"Only by Isolation Forest: {len(only_iso)}")
print(f"Only by Z-Score: {len(only_z)}")

Flagged by both methods: 1
Only by Isolation Forest: 10
Only by Z-Score: 5


**Possible real-world explanations for top anomalies:**
- The **early-January and late-February weeks** flagged as unusually *low*
  (2015-01-04, 2015-02-22, 2016-01-24) line up with the classic post-holiday
  demand lull — fewer orders right after the Nov/Dec peak.
- The **November/December 2018 spikes** (2018-11-04, 2018-11-18, 2018-12-02)
  align with Black-Friday/Cyber-Monday and year-end holiday promotions.
- The **March and September spikes** (2015-03-22, 2015-09-13) likely
  correspond to quarter-end or back-to-school promotional pushes rather than
  the main holiday season — worth confirming against the actual marketing
  calendar for those weeks.

**Do both methods agree?** Isolation Forest and the Z-score method agree on
only 1 of the weeks flagged here (2015-03-22, the biggest single spike in the
series) — they mostly disagree on the rest. Isolation Forest looks at the
overall density of points in a more global sense, while the Z-score method is
purely local (rolling window), so it reacts to sudden local shifts that
Isolation Forest may not flag as unusual against the whole series, and vice
versa. In practice, treating the 1 week both agree on as highest-confidence,
and the rest as "worth a second look" rather than automatic alerts, is a
reasonable way to avoid over-reacting to false positives.

## Task 6 — Product Demand Segmentation using Clustering

In [47]:
subcat_features = df.groupby("Sub-Category").apply(
    lambda g: pd.Series({
        "total_sales": g["Sales"].sum(),
        "avg_order_value": g["Sales"].mean(),
        "sales_volatility": g.groupby(g["Order Date"].dt.to_period("M"))["Sales"].sum().std(),
    })
).reset_index()

# Year-over-year growth rate per sub-category
yoy = df.groupby(["Sub-Category", "Year"])["Sales"].sum().unstack("Year")
yoy_growth = yoy.pct_change(axis=1).mean(axis=1) * 100
subcat_features = subcat_features.merge(yoy_growth.rename("yoy_growth_pct"), on="Sub-Category")
subcat_features = subcat_features.fillna(0)
subcat_features

,Sub-Category,total_sales,avg_order_value,sales_volatility,yoy_growth_pct
0,Accessories,164186.7000,217.178175,2579.994809,37.638071
1,Appliances,104618.4030,227.926804,1821.621539,39.927584
2,Art,26705.4100,34.019631,330.488343,16.605553
3,Binders,200028.7850,134.067550,3848.223648,21.873607
4,Bookcases,113813.1987,503.598224,2220.405080,23.806516
5,Chairs,322822.7310,531.833165,4407.232960,7.135790
6,Copiers,146248.0940,2215.880212,5500.774391,84.671819
7,Envelopes,16128.0460,65.032444,228.218688,-2.766643
8,Fasteners,3001.9600,14.027850,48.742229,15.703383
9,Furnishings,89212.0180,95.823865,1360.017867,28.788938


In [48]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

cluster_cols = ["total_sales", "yoy_growth_pct", "sales_volatility", "avg_order_value"]
X_cluster = subcat_features[cluster_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Elbow method
inertias = []
k_range = range(1, 8)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots()
ax.plot(list(k_range), inertias, marker="o")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Inertia")
ax.set_title("Elbow Method for Optimal K")
plt.tight_layout()
plt.savefig(f"{CHARTS}/elbow_method.png", dpi=120)
plt.close()

In [49]:
OPTIMAL_K = 4  # chosen from the elbow bend
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
subcat_features["cluster"] = kmeans.fit_predict(X_scaled)

cluster_summary = subcat_features.groupby("cluster")[cluster_cols].mean()
print(cluster_summary)

           total_sales  yoy_growth_pct  sales_volatility  avg_order_value
cluster                                                                  
0        167743.362500       46.338910       5552.164569      1930.716763
1         55331.883212       19.958882        907.223743       129.054568
2        239495.780667       18.096484       3489.056075       361.131228
3         46420.308000      192.844843       2025.094139       252.284283


In [50]:
def assign_cluster_labels(cluster_summary):
    """Label each cluster relative to the others (not fixed thresholds), so
    labels stay meaningful and distinct even when, e.g., every sub-category
    happened to grow this period (fixed cutoffs like 'growth > 5%' would
    otherwise label every cluster 'Growing Demand')."""
    cs = cluster_summary.copy()
    cs["sales_rank"] = cs["total_sales"].rank(ascending=False)
    cs["volatility_rank"] = cs["sales_volatility"].rank(ascending=True)  # 1 = most stable
    cs["stability_score"] = cs["sales_rank"] + cs["volatility_rank"]

    labels = {}
    remaining = list(cs.index)

    # 1) Highest volume + most stable combination -> flagship cluster
    c = cs.loc[remaining, "stability_score"].idxmin()
    labels[c] = "High Volume, Stable Demand"
    remaining.remove(c)

    # 2) Fastest growth among what's left
    c = cs.loc[remaining, "yoy_growth_pct"].idxmax()
    labels[c] = "Growing Demand"
    remaining.remove(c)

    # 3) Slowest growth (or actual decline) among what's left
    c = cs.loc[remaining, "yoy_growth_pct"].idxmin()
    labels[c] = "Declining Demand" if cs.loc[c, "yoy_growth_pct"] < 0 else "Slower, Steadier Demand"
    remaining.remove(c)

    # 4) Whatever's left is the volatile long tail
    for c in remaining:
        labels[c] = "Low Volume, High Volatility"

    return labels

cluster_labels = assign_cluster_labels(cluster_summary)
subcat_features["cluster_label"] = subcat_features["cluster"].map(cluster_labels)
subcat_features[["Sub-Category", "cluster", "cluster_label"] + cluster_cols]

,Sub-Category,cluster,cluster_label,total_sales,yoy_growth_pct,sales_volatility,avg_order_value
0,Accessories,2,"Slower, Steadier Demand",164186.7000,37.638071,2579.994809,217.178175
1,Appliances,1,"High Volume, Stable Demand",104618.4030,39.927584,1821.621539,227.926804
2,Art,1,"High Volume, Stable Demand",26705.4100,16.605553,330.488343,34.019631
3,Binders,2,"Slower, Steadier Demand",200028.7850,21.873607,3848.223648,134.067550
4,Bookcases,1,"High Volume, Stable Demand",113813.1987,23.806516,2220.405080,503.598224
5,Chairs,2,"Slower, Steadier Demand",322822.7310,7.135790,4407.232960,531.833165
6,Copiers,0,"Low Volume, High Volatility",146248.0940,84.671819,5500.774391,2215.880212
7,Envelopes,1,"High Volume, Stable Demand",16128.0460,-2.766643,228.218688,65.032444
8,Fasteners,1,"High Volume, Stable Demand",3001.9600,15.703383,48.742229,14.027850
9,Furnishings,1,"High Volume, Stable Demand",89212.0180,28.788938,1360.017867,95.823865


**Note on this run's growth pattern:** every sub-category in this dataset grew
year-over-year (no sub-category had negative growth), so the "Declining
Demand" label above resolves to "Slower, Steadier Demand" instead — the
cluster with the *least* growth relative to the others, not an actual decline.
This is expected and reported honestly rather than forcing a "decline" label
that the data doesn't support.

In [51]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca_coords = pca.fit_transform(X_scaled)
subcat_features["pca1"] = pca_coords[:, 0]
subcat_features["pca2"] = pca_coords[:, 1]

fig, ax = plt.subplots(figsize=(10, 7))
for label in subcat_features["cluster_label"].unique():
    subset = subcat_features[subcat_features["cluster_label"] == label]
    ax.scatter(subset["pca1"], subset["pca2"], label=label, s=120)
    for _, row in subset.iterrows():
        ax.annotate(row["Sub-Category"], (row["pca1"], row["pca2"]), fontsize=8, alpha=0.8)
ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.set_title("Product Sub-Category Demand Segments (PCA projection)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{CHARTS}/cluster_pca.png", dpi=120)
plt.close()

subcat_features.to_csv(f"{CHARTS}/../cluster_assignments.csv", index=False)

**Recommended stocking strategy per cluster:**
- **High Volume, Stable Demand** → keep steady safety stock, reorder on a fixed
  schedule; low forecasting risk.
- **Growing Demand** → increase reorder quantities ahead of trend, monitor
  closely to avoid stockouts as demand ramps up.
- **Declining Demand** (if present) → reduce future purchase orders, consider
  clearance pricing to free up capital and warehouse space.
- **Slower, Steadier Demand** (this run's least-fast-growing cluster, still
  growing overall) → maintain current order volumes, revisit only if growth
  turns negative in a future refresh.
- **Low Volume, High Volatility** → keep lean stock levels, use faster
  replenishment cycles instead of large batch orders, since demand is hard to
  predict precisely for this group.

## Task 7 & 8
The Streamlit dashboard is implemented separately in `app.py` (Task 7), and the
executive business report is `summary.docx` (Task 8) — both included in this
submission folder, built from the same computations performed above.

In [52]:
print("Analysis complete. All charts saved to ./charts/, model comparison saved to "
      "model_comparison.csv, and cluster assignments saved to cluster_assignments.csv")

Analysis complete. All charts saved to ./charts/, model comparison saved to model_comparison.csv, and cluster assignments saved to cluster_assignments.csv
